In [1]:
import numpy as np
import pyshtools as pysh
#from read_grav import grgm1200b
import matplotlib as plt
import boule as bl
import pandas as pd
import xarray as xr
import harmonica as hm
import verde as vd
import pygmt
from pathlib import Path

pygmt-session [ERROR]: Cannot find the PSL_UTF-8 encoding
begin [ERROR]: Cannot find the PSL_UTF-8 encoding
d:\conda_envs\planet\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
test=True
PROJECT_ROOT = Path.cwd().parent
data_filename=PROJECT_ROOT/"data/density_no_mare_n3000_f3050_719.sh"
if test:
    result_filename=PROJECT_ROOT/"data/boueguer_frenquency_11km_1deg.csv"
else:
    result_filename=PROJECT_ROOT/"data/boueguer_frenquency_11km.csv"

In [3]:
def plotgmt(xr):
    with pygmt.config(PROJ_ELLIPSOID=f"{bl.Moon2015.radius}f=0"):
        fig = pygmt.Figure()
        fig.grdimage(
            xr,
            projection="R15c",
            cmap="jet",
            frame="afg",
        )
        fig.colorbar(frame='af+l"Lunar gravity [mGal]"')
        fig.savefig("preview.jpg")
        fig.show()
        


In [4]:
height = 11e3
porosity = 0.12

In [5]:
lmax=719
# 密度
densityfile = data_filename
density = pysh.SHCoeffs.from_file(densityfile, lmax=lmax)
density_grid=density.expand(grid='DH2',lmax=lmax,extend=False)
density_xr=density_grid.to_xarray()
# 重力
pot=pysh.datasets.Moon.GRGM1200B()
pot.set_omega(bl.Moon2015.angular_velocity)
pot=pot.change_ref(gm=bl.Moon2015.geocentric_grav_const, r0=bl.Moon2015.radius)
pot_grid= pot.expand(lmax=lmax, a=bl.Moon2015.radius + height, f=0, normal_gravity=False, extend=False)
pot_xr=pot_grid.to_xarray()
# 地形
topo=pysh.datasets.Moon.LDEM_shape_pa()
topo_grid=topo.expand(grid='DH2',lmax=lmax,extend=False)
topo_xr=topo_grid.to_xarray()

In [6]:
LON, LAT = np.meshgrid(pot_xr.lon, pot_xr.lat) 
Moon2015_el=bl.Moon2015
gamma=Moon2015_el.normal_gravity(latitude=LAT,height=height)

In [7]:
data=-pot_xr.radial.data*1e5-gamma
freeair_xr=xr.DataArray(data,dims=('lat','lon'),coords={'lat': pot_xr.lat,'lon': pot_xr.lon})

In [8]:
bc, r0 = pysh.gravmag.CilmPlusRhoHDH(
    topo_grid.data,
    nmax=8,
    mass=pot.mass,
    rho=density_grid.data * (1 - porosity),
    lmax=lmax
)

In [9]:
bc

array([[[-1.43362488e-05,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 6.22141696e-05, -4.51661921e-04,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [-1.77365023e-04, -2.05776194e-04,  2.82217591e-05, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        ...,
        [-8.14027615e-10,  6.62675449e-11,  4.90335409e-10, ...,
         -5.62342067e-10,  0.00000000e+00,  0.00000000e+00],
        [ 1.95327279e-10,  3.25464363e-10, -1.70151740e-10, ...,
          4.35725393e-10,  9.14748766e-11,  0.00000000e+00],
        [-2.24528466e-10, -2.84619670e-10, -4.98757830e-10, ...,
          3.96415236e-10,  6.54696708e-10,  9.58290163e-10]],

       [[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00, -1.85173494e-04,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e

In [10]:
bc_coff=pysh.SHGravCoeffs.from_array(bc,gm=bl.Moon2015.geocentric_grav_const, r0=bl.Moon2015.radius)

In [11]:
bc_grid=bc_coff.expand(lmax=lmax,a=bl.Moon2015.mean_radius+height,f=0,extend=False)
bc_raw_xr=bc_grid.to_xarray()

In [12]:
bc_xr=xr.DataArray(-bc_raw_xr.radial.data*1e5,dims=('lat','lon'),coords={'lat': pot_xr.lat,'lon': pot_xr.lon})

In [13]:
ba_xr=xr.DataArray(freeair_xr.data-bc_xr.data,dims=('lat','lon'),coords={'lat': pot_xr.lat,'lon': pot_xr.lon})

In [14]:
region=(0, 359.99999, -90, 90)
nlat, nlon = ba_xr.shape
shape = (nlat + 1, nlon + 1)
if test:
    grid_longitude, grid_latitude = vd.grid_coordinates(region=region, shape=(90*2+1,180*2+1))
else:
    grid_longitude, grid_latitude = vd.grid_coordinates(region=region, shape=shape)
lon_o=np.sort(np.unique(grid_longitude))
lat_o=np.sort(np.unique(grid_latitude))
lon = 0.5 * (lon_o[:-1] + lon_o[1:])
lat = 0.5 * (lat_o[:-1] + lat_o[1:])
ba_xr2 = ba_xr.sortby("lat")   # 让 lat 从 -90 -> 90
boueguer_xr_sub=ba_xr2.interp(lat=lat,lon=lon, kwargs={"fill_value": "extrapolate"})
topo_xr2 = topo_xr.sortby("lat")   # 让 lat 从 -90 -> 90
topo_xr_sub=topo_xr2.interp(lat=lat,lon=lon, kwargs={"fill_value": "extrapolate"})
LON, LAT = np.meshgrid(lon, lat) 

In [15]:
df = (
    boueguer_xr_sub
    .stack(points=('lat','lon'))
    .reset_index('points')
    .to_dataframe(name='deltaN')
    .reset_index(drop=True)
)

topo_xr_sub.data = topo_xr_sub.data - bl.Moon2015.mean_radius

df_topo = (
    topo_xr_sub
    .stack(points=('lat','lon'))
    .reset_index('points')
    .to_dataframe(name='topo')
    .reset_index(drop=True)
)

df = df.merge(df_topo, on=['lon', 'lat'], how='left')

df.to_csv(result_filename, index=False)